# 02 Model Experiments

Initial comparison, hyperparameter tuning, ablation, class imbalance variants, and final model selection. `income_test.csv` is not used for model selection in this notebook.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import evaluate_classifier, high_positive_auc_scorer, overfitting_summary
from src.fairness import fairness_gap_table, group_metrics
from src.modeling import build_pipeline, get_model_specs, get_tuning_grids
from src.preprocessing import build_preprocessor, get_feature_groups, split_features_target, validate_training_data
from src.project_paths import INCOME_CSV, OUTPUTS_DIR, REPORTS_DIR

RANDOM_STATE = 42
POSITIVE_LABEL = 'high'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 80)

In [ ]:
train_df = pd.read_csv(INCOME_CSV)
validate_training_data(train_df)
X, y = split_features_target(train_df)
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)
feature_groups = get_feature_groups()
print(f'train split: {X_train.shape}, validation split: {X_val.shape}')

## T5 - Initial model comparison

In [ ]:
initial_rows = []
initial_pipelines = {}
for spec in get_model_specs(random_state=RANDOM_STATE):
    preprocessor = build_preprocessor(feature_groups.numeric, feature_groups.categorical)
    pipeline = build_pipeline(preprocessor, clone(spec.estimator))
    pipeline.fit(X_train[list(feature_groups.all_features)], y_train)
    initial_pipelines[spec.name] = pipeline
    initial_rows.append(evaluate_classifier(pipeline, X_train[list(feature_groups.all_features)], y_train, 'train', model_name=spec.name, variant='full'))
    initial_rows.append(evaluate_classifier(pipeline, X_val[list(feature_groups.all_features)], y_val, 'validation', model_name=spec.name, variant='full'))

model_comparison = pd.concat(initial_rows, ignore_index=True)
model_comparison.to_csv(OUTPUTS_DIR / 'model_comparison.csv', index=False)
display(model_comparison.sort_values(['split', 'auc'], ascending=[True, False]))

## T6 - Hyperparameter tuning and overfitting analysis

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
grids = get_tuning_grids()
tuning_rows = []
tuned_metric_rows = []
tuned_pipelines = {}

for spec in get_model_specs(random_state=RANDOM_STATE):
    preprocessor = build_preprocessor(feature_groups.numeric, feature_groups.categorical)
    pipeline = build_pipeline(preprocessor, clone(spec.estimator))
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=grids[spec.name],
        scoring=high_positive_auc_scorer,
        cv=cv,
        n_jobs=1,
        refit=True,
        return_train_score=True,
    )
    search.fit(X_train[list(feature_groups.all_features)], y_train)
    tuned_pipelines[spec.name] = search.best_estimator_

    tuning_rows.append({
        'model': spec.name,
        'variant': 'tuned_full',
        'best_cv_auc': search.best_score_,
        'best_params': search.best_params_,
    })
    tuned_metric_rows.append(evaluate_classifier(search.best_estimator_, X_train[list(feature_groups.all_features)], y_train, 'train', model_name=spec.name, variant='tuned_full'))
    tuned_metric_rows.append(evaluate_classifier(search.best_estimator_, X_val[list(feature_groups.all_features)], y_val, 'validation', model_name=spec.name, variant='tuned_full'))

tuning_results = pd.DataFrame(tuning_rows)
tuned_metrics = pd.concat(tuned_metric_rows, ignore_index=True)
overfit = overfitting_summary(
    tuned_metrics[tuned_metrics['split'] == 'train'],
    tuned_metrics[tuned_metrics['split'] == 'validation'],
)
tuning_results.to_csv(OUTPUTS_DIR / 'tuning_results.csv', index=False)
overfit.to_csv(OUTPUTS_DIR / 'overfitting_summary.csv', index=False)
display(tuning_results)
display(overfit.sort_values('auc_gap'))

## T7 - Feature ablation experiments

In [ ]:
feature_variants = {
    'full': get_feature_groups(),
    'high_missing_removed': get_feature_groups(include_high_missing=False),
    'sex_removed': get_feature_groups(include_sex=False),
}
base_specs = {spec.name: spec for spec in get_model_specs(random_state=RANDOM_STATE)}
ablation_rows = []

for model_name, tuned_pipeline in tuned_pipelines.items():
    best_estimator = clone(base_specs[model_name].estimator)
    best_estimator.set_params(**{
        key.replace('model__', ''): value
        for key, value in tuned_pipeline.get_params().items()
        if key.startswith('model__') and '__' not in key.replace('model__', '')
    })
    for variant_name, groups in feature_variants.items():
        preprocessor = build_preprocessor(groups.numeric, groups.categorical)
        pipeline = build_pipeline(preprocessor, clone(best_estimator))
        pipeline.fit(X_train[list(groups.all_features)], y_train)
        ablation_rows.append(evaluate_classifier(pipeline, X_val[list(groups.all_features)], y_val, 'validation', model_name=model_name, variant=variant_name))

ablation_summary = pd.concat(ablation_rows, ignore_index=True)
ablation_summary.to_csv(OUTPUTS_DIR / 'ablation_summary.csv', index=False)
display(ablation_summary.sort_values('auc', ascending=False))

## T8 - Class imbalance variants and final selection

In [ ]:
imbalance_specs = [
    ('Logistic Regression', LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=RANDOM_STATE)),
    ('Random Forest', RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=5, max_features='sqrt', class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)),
]
imbalance_rows = []
imbalance_pipelines = {}

for model_name, estimator in imbalance_specs:
    preprocessor = build_preprocessor(feature_groups.numeric, feature_groups.categorical)
    pipeline = build_pipeline(preprocessor, estimator)
    pipeline.fit(X_train[list(feature_groups.all_features)], y_train)
    imbalance_pipelines[model_name] = pipeline
    imbalance_rows.append(evaluate_classifier(pipeline, X_train[list(feature_groups.all_features)], y_train, 'train', model_name=model_name, variant='class_weight_balanced'))
    imbalance_rows.append(evaluate_classifier(pipeline, X_val[list(feature_groups.all_features)], y_val, 'validation', model_name=model_name, variant='class_weight_balanced'))

class_imbalance_summary = pd.concat(imbalance_rows, ignore_index=True)
class_imbalance_summary.to_csv(OUTPUTS_DIR / 'class_imbalance_summary.csv', index=False)
display(class_imbalance_summary.sort_values(['split', 'auc'], ascending=[True, False]))

In [ ]:
candidate_summary = pd.concat([
    model_comparison,
    tuned_metrics,
    ablation_summary,
    class_imbalance_summary,
], ignore_index=True)
validation_candidates = candidate_summary[candidate_summary['split'] == 'validation'].copy()
validation_candidates = validation_candidates.sort_values(
    ['auc', 'recall_high', 'precision_high', 'accuracy'],
    ascending=[False, False, False, False],
)
validation_candidates = validation_candidates.drop_duplicates(['model', 'variant'], keep='first').reset_index(drop=True)
validation_candidates.to_csv(OUTPUTS_DIR / 'final_candidate_summary.csv', index=False)
best_candidate = validation_candidates.iloc[0]
display(validation_candidates.head(10))
print('Selected final validation candidate:', best_candidate['model'], best_candidate['variant'])

In [ ]:
def pipeline_for_candidate(row):
    model_name = row['model']
    variant = row['variant']
    if variant == 'tuned_full':
        return tuned_pipelines[model_name], feature_variants['full']
    if variant in feature_variants:
        groups = feature_variants[variant]
        source = tuned_pipelines[model_name]
        estimator = clone(source.named_steps['model'])
        preprocessor = build_preprocessor(groups.numeric, groups.categorical)
        pipeline = build_pipeline(preprocessor, estimator)
        pipeline.fit(X_train[list(groups.all_features)], y_train)
        return pipeline, groups
    if variant == 'class_weight_balanced':
        return imbalance_pipelines[model_name], feature_variants['full']
    return initial_pipelines[model_name], feature_variants['full']

final_pipeline, final_groups = pipeline_for_candidate(best_candidate)
y_val_pred = final_pipeline.predict(X_val[list(final_groups.all_features)])
y_val_proba = final_pipeline.predict_proba(X_val[list(final_groups.all_features)])
fairness_metrics = group_metrics(
    y_val,
    y_val_pred,
    y_val_proba,
    X_val['sex'],
    classes=final_pipeline.classes_,
)
fairness_gaps = fairness_gap_table(fairness_metrics)
fairness_metrics.insert(0, 'model', best_candidate['model'])
fairness_metrics.insert(1, 'variant', best_candidate['variant'])
fairness_metrics.to_csv(OUTPUTS_DIR / 'fairness_metrics.csv', index=False)

fairness_comparison_rows = []
for comparison_variant in ['full', 'sex_removed']:
    comparison_row = pd.Series({'model': best_candidate['model'], 'variant': comparison_variant})
    comparison_pipeline, comparison_groups = pipeline_for_candidate(comparison_row)
    comparison_pred = comparison_pipeline.predict(X_val[list(comparison_groups.all_features)])
    comparison_proba = comparison_pipeline.predict_proba(X_val[list(comparison_groups.all_features)])
    comparison_group_metrics = group_metrics(
        y_val,
        comparison_pred,
        comparison_proba,
        X_val['sex'],
        classes=comparison_pipeline.classes_,
    )
    comparison_gap = fairness_gap_table(comparison_group_metrics)
    for _, gap_row in comparison_gap.iterrows():
        fairness_comparison_rows.append({
            'model': best_candidate['model'],
            'variant': comparison_variant,
            'metric': gap_row['metric'],
            'gap': gap_row['gap'],
        })
fairness_ablation_comparison = pd.DataFrame(fairness_comparison_rows)
fairness_ablation_comparison.to_csv(OUTPUTS_DIR / 'fairness_ablation_comparison.csv', index=False)
display(fairness_metrics)
display(fairness_gaps)
display(fairness_ablation_comparison)

In [ ]:
notes_path = REPORTS_DIR / 'report_notes_tr.md'
memo_path = REPORTS_DIR / 'final_model_selection_memo_tr.md'

tuned_best = validation_candidates.iloc[0]
overfit_best = overfit.sort_values('auc_gap').iloc[0]
sex_removed = ablation_summary[ablation_summary['variant'] == 'sex_removed'].sort_values('auc', ascending=False).iloc[0]
high_missing_removed = ablation_summary[ablation_summary['variant'] == 'high_missing_removed'].sort_values('auc', ascending=False).iloc[0]
balanced_best = class_imbalance_summary[class_imbalance_summary['split'] == 'validation'].sort_values('auc', ascending=False).iloc[0]
positive_gap = fairness_gaps[fairness_gaps['metric'] == 'positive_prediction_rate']['gap'].iloc[0]
overfit_gap_summary = '; '.join(
    f"{row['model']}: AUC gap {row['auc_gap']:.3f}"
    for _, row in overfit.sort_values('model').iterrows()
)
full_fairness_gap = fairness_ablation_comparison[
    (fairness_ablation_comparison['variant'] == 'full')
    & (fairness_ablation_comparison['metric'] == 'positive_prediction_rate')
]['gap'].iloc[0]
sex_removed_fairness_gap = fairness_ablation_comparison[
    (fairness_ablation_comparison['variant'] == 'sex_removed')
    & (fairness_ablation_comparison['metric'] == 'positive_prediction_rate')
]['gap'].iloc[0]

appendix = f'''
## T6 - Hyperparameter tuning ve overfitting

- Tuning (hyperparameter tuning) sadece training split uzerinde 3-fold StratifiedKFold ile yapildi; validation split final karsilastirma icin ayrik tutuldu.
- En iyi validation AUC {tuned_best['model']} / {tuned_best['variant']} icin {tuned_best['auc']:.3f} olarak goruldu.
- AUC train-validation gap ozeti: {overfit_gap_summary}.

## T7 - Feature ablation

- High-missing kolonlari cikarilan en iyi varyant {high_missing_removed['model']} ile validation AUC {high_missing_removed['auc']:.3f} verdi.
- `sex` cikarilan en iyi varyant {sex_removed['model']} ile validation AUC {sex_removed['auc']:.3f} verdi.
- Final model ailesinde positive prediction rate gap full icin {full_fairness_gap:.3f}, sex removed icin {sex_removed_fairness_gap:.3f}; bu fairness karsilastirmasi icin ayrica saklandi.
- Bu sonuclar feature selection (ozellik secimi) ve fairness tartismasi icin kanit olarak saklandi.

## T8 - Class imbalance ve final model selection

- Class weight balanced varyantlarinda en iyi validation AUC {balanced_best['model']} icin {balanced_best['auc']:.3f} oldu.
- Balanced varyant {balanced_best['model']} high recall {balanced_best['recall_high']:.3f} ile daha yuksek recall verdi, ancak final aday AUC dengesinde daha iyi kaldi.
- Final aday {best_candidate['model']} / {best_candidate['variant']} olarak secildi; validation AUC {best_candidate['auc']:.3f}, high recall {best_candidate['recall_high']:.3f}.
- Final aday icin validation setinde sex bazli positive prediction rate gap {positive_gap:.3f}; fairness (adalet) tartismasinda bu sayi kullanilacak.
- `income_test.csv` model secimi, tuning veya threshold belirleme icin kullanilmadi.
'''

existing_notes = notes_path.read_text(encoding='utf-8') if notes_path.exists() else '# Rolling Report Notes\n'
base_notes = existing_notes.split('\n## T6 - Hyperparameter tuning ve overfitting')[0].rstrip()
notes_path.write_text(base_notes + '\n' + appendix, encoding='utf-8')

memo = f'''# Final Model Selection Memo - Turkish Draft

## Secilen aday

Final aday: **{best_candidate['model']} / {best_candidate['variant']}**.

## Sayisal gerekce

- Birincil metrik (primary metric) validation AUC idi: {best_candidate['auc']:.3f}.
- Accuracy: {best_candidate['accuracy']:.3f}.
- High precision/recall: {best_candidate['precision_high']:.3f} / {best_candidate['recall_high']:.3f}.
- Low precision/recall: {best_candidate['precision_low']:.3f} / {best_candidate['recall_low']:.3f}.

## Overfitting ve feature selection yorumu

Tuning (hyperparameter tuning) sadece training split uzerinde cross-validation ile yapildi. Validation split model secimi icin ayrik tutuldu. Train-validation AUC gap ozeti: {overfit_gap_summary}. Feature ablation deneylerinde full, high-missing removed ve sex removed varyantlari karsilastirildi. Son secim validation AUC, precision/recall dengesi, train-validation gap ve fairness gap birlikte dusunulerek yapildi.

## Class imbalance yorumu

Class weight balanced varyantlarinda en iyi validation AUC {balanced_best['model']} icin {balanced_best['auc']:.3f} oldu. Bu varyant high recall {balanced_best['recall_high']:.3f} ile final adaydan yuksek recall verdi, fakat precision ve AUC dengesi daha zayif kaldigi icin final secim yapilmadi.

## Fairness notu

Final aday icin sex bazli positive prediction rate gap {positive_gap:.3f}. Final model ailesinde full gap {full_fairness_gap:.3f}, sex removed gap {sex_removed_fairness_gap:.3f}. Bu, fairness (adalet) acisindan fark oldugunu ve sadece `sex` kolonunu cikarmanin fairness'i garanti etmedigini gosterir.

## Guardrail

`income_test.csv` final model secimi, tuning veya threshold belirleme icin kullanilmadi. Test accuracy iddia edilmeyecek; performans yorumu validation/CV sonuclarina dayanacak.
'''

memo_path.write_text(memo, encoding='utf-8')
print(memo_path)